# 🦙  — Test Case Generator
**Đề tài:** Phân tích hiệu quả của LLM trong quy trình phát triển phần mềm

**Hướng dẫn sử dụng:**
1. Vào `Runtime` → `Change runtime type` → chọn **T4 GPU**
2. Chạy từng cell theo thứ tự (Shift+Enter)
3. Đến Cell 4: nhập User Story + AC → nhấn Run → nhận test case output

> ⚠️ Lần đầu chạy Cell 2 sẽ mất ~5-10 phút để download model (~5GB)

## Cell 1 — Cài đặt thư viện

In [4]:
# Cài đặt các thư viện cần thiết
!pip install -q transformers accelerate bitsandbytes torch
!pip install -q huggingface_hub

import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
else:
    print('⚠️ Không có GPU! Vào Runtime > Change runtime type > T4 GPU')

GPU available: True
GPU: Tesla T4
VRAM: 14.6 GB


In [ ]:
# CELL MỚI — Dọn VRAM trước khi load model mới
import torch, gc

# Xóa model cũ nếu còn trong memory
try:
    del model
    del tokenizer
    print('Đã xóa model cũ')
except NameError:
    print('Không có model cũ')

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

print(f'VRAM free: {torch.cuda.mem_get_info()[0]/1024**3:.2f} GB / '
      f'{torch.cuda.mem_get_info()[1]/1024**3:.2f} GB')

Không có model cũ
VRAM free: 14.46 GB / 14.56 GB


## Cell 2 — Load model
> ⏳ Lần đầu chạy mất ~5-10 phút để download (~5GB). Các lần sau nhanh hơn vì đã cache.

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'

# Quantize 4-bit để fit trong T4 16GB VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16
)

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print('Loading model (4-bit quantized)...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True
)

print('✅ Model loaded thành công!')
print(f'Memory used: {torch.cuda.memory_allocated()/1024**3:.1f} GB / {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')

Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


Loading model (4-bit quantized)...


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


✅ Model loaded thành công!
Memory used: 2.1 GB / 14.6 GB


## Cell 3 — Hàm sinh test case
> Chạy 1 lần để định nghĩa hàm. Không cần sửa.

In [2]:
from IPython.display import display, Markdown
import time

SYSTEM_PROMPT = """Bạn là một QA Engineer chuyên nghiệp. Viết bộ test case theo format bảng markdown 7 cột:
ID | Test Case Description | Category | Pre-condition / Test Data | Test Step | Expected Result | Priority

Quy tắc:
- ID: PREFIX_001 (prefix viết HOA từ tên feature)
- Test Step: đánh số 1. 2. 3., bắt đầu bằng động từ
- Expected Result: đánh số tương ứng
- Priority: High / Medium / Low
- Coverage: Positive, Negative, Boundary, Edge case, Security
- Viết bằng Tiếng Việt
- TRƯỚC bảng: viết ## Coverage Summary"""


def generate_test_cases(user_story, acceptance_criteria,
                        feature_name='Feature', figma='Không có',
                        api_docs='Không có', max_new_tokens=4096):

    full_prompt = f"""{SYSTEM_PROMPT}

---
Feature: {feature_name}

User Story:
{user_story}

Acceptance Criteria:
{acceptance_criteria}

Figma: {figma}
API docs: {api_docs}
---
Hãy sinh bộ test case đầy đủ:"""

    # Dùng tokenizer thông thường, KHÔNG dùng apply_chat_template
    inputs = tokenizer(full_prompt, return_tensors='pt').to(model.device)
    input_length = inputs['input_ids'].shape[-1]

    print(f'⏳ Đang sinh test case cho: {feature_name}...')
    start = time.time()

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.3,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    elapsed = time.time() - start
    new_tokens = output_ids[0][input_length:]
    result = tokenizer.decode(new_tokens, skip_special_tokens=True)

    print(f'✅ Hoàn thành trong {elapsed:.1f}s | {len(new_tokens)} tokens')
    return result


print('✅ Sẵn sàng!')

✅ Sẵn sàng!


In [ ]:
# DEBUG: kiểm tra apply_chat_template trả về gì
test_messages = [{"role": "user", "content": "hello"}]
test_out = tokenizer.apply_chat_template(test_messages, add_generation_prompt=True, return_tensors='pt')
print(type(test_out))
print(test_out)

<class 'transformers.tokenization_utils_base.BatchEncoding'>
{'input_ids': tensor([[32010, 22172, 32007, 32001]]), 'attention_mask': tensor([[1, 1, 1, 1]])}


## Cell 4 — ✏️ NHẬP INPUT VÀ CHẠY
> **Chỉnh sửa phần này.** Nhập User Story + AC của feature bạn muốn test vào đây, rồi nhấn Shift+Enter.

In [3]:
# ============================================================
# ✏️  CHỈNH SỬA PHẦN NÀY
# ============================================================

FEATURE_NAME = "Đăng nhập / Login"

USER_STORY = """
As a registered user,
I want to log in to the Ticketbox app using my email and password,
So that I can access my account and purchase tickets.
"""

ACCEPTANCE_CRITERIA = """
Scenario 1: Đăng nhập thành công
  Given user đã có tài khoản hợp lệ
  When user nhập đúng email và password
  Then user được redirect về màn hình Home
  And hiển thị tên user ở header

Scenario 2: Email không tồn tại
  Given user nhập email chưa đăng ký
  When user submit form
  Then hiển thị lỗi "Email không tồn tại"

Scenario 3: Sai password
  Given user nhập đúng email nhưng sai password
  When user submit form
  Then hiển thị lỗi "Mật khẩu không chính xác"
  And sau 5 lần sai liên tiếp, tài khoản bị khóa tạm thời 15 phút

Scenario 4: Bỏ trống field
  Given user không nhập email hoặc password
  When user tap nút Đăng nhập
  Then hiển thị validation error tại field bị trống
  And nút Đăng nhập bị disabled khi chưa nhập đủ

Scenario 5: Quên mật khẩu
  Given user ở màn hình Login
  When user tap "Quên mật khẩu?"
  Then redirect sang màn hình Reset Password
"""

FIGMA_LINK = "Không có"
API_DOCS = "Không có"

# ============================================================
# 🚀 CHẠY — không cần sửa từ đây trở xuống
# ============================================================
output = generate_test_cases(
    user_story=USER_STORY,
    acceptance_criteria=ACCEPTANCE_CRITERIA,
    feature_name=FEATURE_NAME,
    figma=FIGMA_LINK,
    api_docs=API_DOCS
)

display(Markdown(output))

⏳ Đang sinh test case cho: Đăng nhập / Login...


AttributeError: 'DynamicCache' object has no attribute 'seen_tokens'

## Cell 5 — Lưu output ra file
> Chạy cell này để lưu kết quả thành file `.md` trong Colab, rồi download về máy.

In [ ]:
import re
from google.colab import files

# Tạo tên file từ feature name
safe_name = re.sub(r'[^\w\s-]', '', FEATURE_NAME).strip().replace(' ', '_')
filename = f'TC_Llama_{safe_name}.md'

with open(filename, 'w', encoding='utf-8') as f:
    f.write(f'# Test Cases — {FEATURE_NAME}\n')
    f.write(f'**Model:** Llama 3.1 8B (4-bit quantized on Google Colab T4)\n')
    f.write(f'**Feature:** {FEATURE_NAME}\n\n')
    f.write('---\n\n')
    f.write(output)

print(f'✅ Đã lưu: {filename}')
files.download(filename)
print('📥 File đang được download về máy...')

✅ Đã lưu: TC_Llama_Đăng_nhập__Login.md


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 File đang được download về máy...


## Cell 6 — (Tùy chọn) Chạy nhiều feature liên tiếp
> Dùng khi cần test nhiều AC khác nhau trong 1 session để tiết kiệm thời gian load model.

In [ ]:
# Thêm các feature vào list này
BATCH_FEATURES = [
    {
        'feature_name': 'Đăng nhập / Login',
        'user_story': USER_STORY,       # dùng lại biến từ Cell 4
        'acceptance_criteria': ACCEPTANCE_CRITERIA,
        'figma': 'Không có',
        'api_docs': 'Không có'
    },
    # Thêm feature 2, 3... theo cùng format ở đây
]

all_results = {}

for item in BATCH_FEATURES:
    print(f"\n{'='*60}")
    print(f"Feature: {item['feature_name']}")
    print('='*60)

    result = generate_test_cases(
        user_story=item['user_story'],
        acceptance_criteria=item['acceptance_criteria'],
        feature_name=item['feature_name'],
        figma=item.get('figma', 'Không có'),
        api_docs=item.get('api_docs', 'Không có')
    )
    all_results[item['feature_name']] = result
    display(Markdown(f"### Output: {item['feature_name']}"))
    display(Markdown(result))

# Lưu tất cả vào 1 file
with open('TC_Llama_Batch.md', 'w', encoding='utf-8') as f:
    for name, content in all_results.items():
        f.write(f'# {name}\n\n')
        f.write(content)
        f.write('\n\n---\n\n')

from google.colab import files
files.download('TC_Llama_Batch.md')
print('\n✅ Batch hoàn thành! File đang được download...')


Feature: Đăng nhập / Login
⏳ Đang sinh test case cho: Đăng nhập / Login...
✅ Hoàn thành trong 90.8s | 952 tokens


### Output: Đăng nhập / Login

 ## Coverage Summary
- Positive: 1, 2, 3, 4, 5
- Negative: 2, 3, 4
- Boundary: 1, 2, 3, 4, 5
- Edge case: 3
- Security: 3

| ID          | Test Case Description                    | Category     | Pre-condition / Test Data                | Test Step            | Expected Result       | Priority |
|-------------|------------------------------------------|--------------|-------------------------------------------|----------------------|-----------------------|----------|
| LOGIN_001   | Đăng nhập thành công                    | Positive     | User đã có tài khoản hợp lệ               | 1. Nhập đúng email và password | Được redirect về màn hình Home và hiển thị tên user ở header | High     |
| LOGIN_002   | Đăng nhập với email không tồn tại        | Negative     | User nhập email chưa đăng ký              | 1. Nhập email chưa đăng ký    | Hiển thị lỗi "Email không tồn tại"                        | High     |
| LOGIN_003   | Đăng nhập với sai password               | Negative     | User nhập đúng email nhưng sai password    | 1. Nhập đúng email, sai password | Hiển thị lỗi "Mật khẩu không chính xác", tài khoản bị khóa tạm thời 15 phút | High     |
| LOGIN_004   | Đăng nhập bỏ trống field                 | Negative     | User không nhập email hoặc password       | 1. Không nhập email hoặc password | Hiển thị validation error tại field bị trống, nút Đăng nhập bị disabled | High     |
| LOGIN_005   | Đăng nhập quên mật khẩu                  | Positive     | User ở màn hình Login                     | 1. Tap "Quên mật khẩu?"      | Redirect sang màn hình Reset Password                       | High     |

## Coverage Summary
- Positive: LOGIN_001, LOGIN_005
- Negative: LOGIN_002, LOGIN_003, LOGIN_004
- Boundary: LOGIN_001, LOGIN_002, LOGIN_003, LOGIN_004, LOGIN_005
- Edge case: LOGIN_003
- Security: LOGIN_003

Lưu ý: ID của các test case đã được định dạng theo yêu cầu, và Priority được gán dựa trên mức độ quan trọng của từng scenario. Các trường hợp boundary và edge case đã được bao gồm trong các test case tương ứng. ## Coverage Summary
- Positive: LOGIN_001, LOGIN_005
- Negative: LOGIN_002, LOGIN_003, LOGIN_004
- Boundary: LOGIN_001, LOGIN_002, LOGIN_003, LOGIN_004, LOGIN_005
- Edge case: LOGIN_003
- Security: LOGIN_003

| ID          | Test Case Description                    | Category     | Pre-condition / Test Data                | Test Step            | Expected Result       | Priority |
|-------------|------------------------------------------|--------------|-------------------------------------------|----------------------|-----------------------|----------|
| LOGIN_001   | Đăng nhập thành công                    | Positive     | User đã có tài khoản hợp lệ               | 1. Nhập đúng email và password | Được redirect về màn hình Home và hiển thị tên user ở header | High     |
| LOGIN_002   | Đăng nhập với email không tồn tại        | Negative     | User nhập email chưa đăng ký              | 1. Nhập email chưa đăng ký    | Hiển thị lỗi "Email không tồn tại"                        | High     |
| LOGIN_003   | Đăng nhập với sai password               | Negative     | User nhập đúng email nhưng sai password    | 1. Nhập đúng email, sai password | Hiển thị lỗi "Mật khẩu không chính xác", tài khoản bị khóa tạm thời 15 phút | High     |
| LOGIN_004   | Đăng nhập bỏ trống field                 | Negative     | User không nhập email hoặc password       | 1. Không nhập email hoặc password | Hiển thị validation error tại field bị trống, nút Đăng nhập bị disabled | High     |
| LOGIN_005   | Đăng nhập quên mật khẩu                  | Positive     | User ở màn hình Login                     | 1. Tap "Quên mật khẩu?"      | Redirect sang màn hình Reset Password                       | High     |

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Batch hoàn thành! File đang được download...


---
## 📋 Hướng dẫn dùng cho đề tài

### Quy trình thực nghiệm so sánh 3 LLM:

| Bước | Mô tả |
|---|---|
| 1 | Chọn 3-5 User Story/AC thực tế (benchmark dataset) |
| 2 | Chạy notebook này → lấy output Llama 3.1 8B |
| 3 | Gửi cùng prompt cho ChatGPT (web/API) → lưu output |
| 4 | Gửi cùng prompt cho Gemini (Google AI Studio) → lưu output |
| 5 | Chấm điểm 3 output theo AcceptanceChecklist.md (thang 10 điểm) |
| 6 | Tổng hợp bảng so sánh: điểm số + chi phí + thời gian sinh |

### Metrics cần ghi lại cho mỗi output:
- **Điểm chất lượng** (X/10 theo AcceptanceChecklist)
- **Số test case sinh ra**
- **Coverage rate** (% AC scenario được cover)
- **Edge case ratio** (% TC là edge/boundary/negative)
- **Thời gian sinh** (giây)
- **Chi phí** (Llama = $0, GPT-4o = ~$0.01-0.03/request, Gemini = free tier)

### Lưu ý khi dùng Llama:
- Nếu Colab ngắt session, chạy lại từ Cell 2 (model cần load lại)
- Llama 3.1 8B tiếng Việt không mạnh bằng GPT-4o — đây là điểm so sánh quan trọng
- Nếu muốn kết quả tiếng Việt tốt hơn, thử thêm `'Qwen/Qwen2.5-7B-Instruct'` thay MODEL_ID

### Thay thế MODEL_ID nếu muốn so sánh thêm:
```python
# Các model thay thế (đều chạy được trên T4 16GB)
MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'        # Tốt cho tiếng Việt hơn
MODEL_ID = 'mistralai/Mistral-7B-Instruct-v0.3'  # Nhẹ, nhanh hơn
MODEL_ID = 'google/gemma-2-9b-it'             # Google, cân bằng
```